In [ ]:
!pip install nltk

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re


from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,MultiLabelBinarizer
from sklearn.metrics import classification_report,hamming_loss


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,Bidirectional,GRU,SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

nltk.download('stopwords',quiet=True)
nltk.download('wordnet',quiet=True)
nltk.download('punkt_tab',quiet=True)
nltk.download('punkt',quiet=True)

True

In [2]:
df = pd.read_csv("/content/train.csv")
df.shape
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
lemmatizer=WordNetLemmatizer()
stop_words=set(stopwords.words('english'))

def clean_text(text):
  text=re.sub(r'[^a-zA-Z]',' ',text)
  text=text.lower()
  text=word_tokenize(text)
  text=[lemmatizer.lemmatize(word) for word in text if word not in stop_words]
  text=' '.join(text)
  return(text)

df['cleaned_comment_text'] = df['comment_text'].apply(clean_text)
df['cleaned_comment_text']

,cleaned_comment_text
0,explanation edits made username hardcore metal...
1,aww match background colour seemingly stuck th...
2,hey man really trying edit war guy constantly ...
3,make real suggestion improvement wondered sect...
4,sir hero chance remember page
...,...
159566,second time asking view completely contradicts...
159567,ashamed horrible thing put talk page
159568,spitzer umm there actual article prostitution ...
159569,look like actually put speedy first version de...


In [ ]:
vocab_len.sum()

np.int64(5535193)

In [ ]:
vocab_len.max()

1250

In [ ]:
vocab_len.shape

(159571,)

In [ ]:
lengths = [len(text.split()) for text in df['cleaned_comment_text']]
import numpy as np

print(np.mean(lengths))
print(np.max(lengths))
print(np.percentile(lengths, 95))

34.68796335173685
1250
117.0


In [4]:
Vocb_Size=5000
Max_len=120

tokenizer=Tokenizer(num_words=Vocb_Size,oov_token='<OOV>')
tokenizer.fit_on_texts(df['cleaned_comment_text'])


sequence=tokenizer.texts_to_sequences(df['cleaned_comment_text'])

X=pad_sequences(sequence, maxlen=Max_len,padding='post')
X

array([[ 467,   56,   58, ...,    0,    0,    0],
       [   1,  966,  423, ...,    0,    0,    0],
       [ 293,  300,   62, ...,    0,    0,    0],
       ...,
       [   1,    1, 4475, ...,    0,    0,    0],
       [  49,    9,  126, ...,    0,    0,    0],
       [  62,   12,  159, ...,    0,    0,    0]], dtype=int32)

In [ ]:
X

array([[ 467,   56,   58, ...,    0,    0,    0],
       [   1,  966,  423, ...,    0,    0,    0],
       [ 293,  300,   62, ...,    0,    0,    0],
       ...,
       [   1,    1, 4475, ...,    0,    0,    0],
       [  49,    9,  126, ...,    0,    0,    0],
       [  62,   12,  159, ...,    0,    0,    0]], dtype=int32)

In [ ]:
df.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate', 'cleaned_comment_text'],
      dtype='object')

In [5]:
y = df[['toxic','severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate']]

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
y_train.shape[1]

6

In [7]:
Num_Topic = y_train.shape[1]

topic_model=Sequential(
    [Embedding(Vocb_Size,64,input_length=Max_len),
    Bidirectional(LSTM(128)),
    Dropout(0.3),
    Dense(64,activation='relu'),
    Dropout(0.2),
    Dense(Num_Topic,activation='sigmoid')]
)

topic_model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
topic_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = EarlyStopping(
    monitor              = 'val_loss',
    patience             = 3,
    restore_best_weights = True
)

history_topic = topic_model.fit(
    X_train, y_train,
    validation_data = (X_test, y_test),
    epochs          = 20,
    batch_size      = 32,
    callbacks       = [early_stop],
    verbose         = 1
)


Epoch 1/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 75s 17ms/step - accuracy: 0.9212 - loss: 0.0659 - val_accuracy: 0.9941 - val_loss: 0.0520
Epoch 2/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 67s 17ms/step - accuracy: 0.9882 - loss: 0.0503 - val_accuracy: 0.9941 - val_loss: 0.0499
Epoch 3/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 69s 17ms/step - accuracy: 0.9850 - loss: 0.0464 - val_accuracy: 0.9939 - val_loss: 0.0504
Epoch 4/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 66s 17ms/step - accuracy: 0.9890 - loss: 0.0427 - val_accuracy: 0.9941 - val_loss: 0.0517
Epoch 5/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 66s 17ms/step - accuracy: 0.9858 - loss: 0.0397 - val_accuracy: 0.9939 - val_loss: 0.0520


In [ ]:
topic_model_RNN=Sequential(
    [Embedding(Vocb_Size,64,input_length=Max_len),
    Bidirectional(SimpleRNN(128)),
    Dropout(0.3),
    Dense(64,activation='relu'),
    Dropout(0.2),
    Dense(Num_Topic,activation='sigmoid')]
)

topic_model_RNN.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


In [ ]:
history_topic_RNN = topic_model_RNN.fit(
    X_train, y_train,
    validation_data = (X_test, y_test),
    epochs          = 20,
    batch_size      = 32,
    callbacks       = [early_stop],
    verbose         = 1
)

Epoch 1/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 85s 21ms/step - accuracy: 0.9926 - loss: 0.0696 - val_accuracy: 0.9941 - val_loss: 0.0693
Epoch 2/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 85s 21ms/step - accuracy: 0.9929 - loss: 0.0640 - val_accuracy: 0.9941 - val_loss: 0.0637
Epoch 3/20
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 85s 21ms/step - accuracy: 0.9924 - loss: 0.0651 - val_accuracy: 0.9941 - val_loss: 0.0774


In [10]:
test_df = pd.read_csv('/content/test.csv')
test_comments = test_df['comment_text']
test_comments

,comment_text
0,Yo bitch Ja Rule is more succesful then you'll...
1,== From RfC == \n\n The title is fine as it is...
2,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,":If you have a look back at the source, the in..."
4,I don't anonymously edit articles at all.
...,...
153159,". \n i totally agree, this stuff is nothing bu..."
153160,== Throw from out field to home plate. == \n\n...
153161,""" \n\n == Okinotorishima categories == \n\n I ..."
153162,""" \n\n == """"One of the founding nations of the..."


In [13]:
test_df['cleaned_test_comments'] = test_df['comment_text'].apply(clean_text)
test_df['cleaned_test_comments']

,cleaned_test_comments
0,yo bitch ja rule succesful ever whats hating s...
1,rfc title fine imo
2,source zawe ashton lapland
3,look back source information updated correct f...
4,anonymously edit article
...,...
153159,totally agree stuff nothing long crap
153160,throw field home plate get faster throwing cut...
153161,okinotorishima category see change agree corre...
153162,one founding nation eu germany law return quit...


In [15]:

sequence=tokenizer.texts_to_sequences(test_df['cleaned_test_comments'])

transfomed_text_comments =pad_sequences(sequence, maxlen=Max_len,padding='post')

In [16]:
transfomed_text_comments

array([[2235,  412,    1, ...,    0,    0,    0],
       [ 987,  187,  543, ...,    0,    0,    0],
       [  10,    1,    1, ...,    0,    0,    0],
       ...,
       [   1,  246,   11, ...,    0,    0,    0],
       [   6,    1,  806, ...,    0,    0,    0],
       [  93,  139,  746, ...,    0,    0,    0]], dtype=int32)

In [17]:
test_prediction = topic_model.predict(transfomed_text_comments)

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 26s 5ms/step


In [18]:
y.columns

Index(['toxic', 'severe_toxic', 'obscene', 'threat', 'insult',
       'identity_hate'],
      dtype='object')

In [20]:
# Predict probabilities/classes
test_prediction = topic_model.predict(transfomed_text_comments)

# Class names
class_names = y.columns.tolist()

# Convert predictions into readable labels
predicted_classes = []

for pred in test_prediction:

    labels = [
        class_names[i]
        for i, value in enumerate(pred)
        if value == 1
    ]

    # If no class predicted
    if len(labels) == 0:
        labels = ['clean']

    predicted_classes.append(labels)


# Display predictions
for text, labels in zip(test_comments[:5], predicted_classes[:5]):

    print("TEXT:")
    print(text)

    print("\nPREDICTED CLASSES:")
    print(labels)

    print("-" * 50)

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 25s 5ms/step
TEXT:
Yo bitch Ja Rule is more succesful then you'll ever be whats up with you and hating you sad mofuckas...i should bitch slap ur pethedic white faces and get you to kiss my ass you guys sicken me. Ja rule is about pride in da music man. dont diss that shit on him. and nothin is wrong bein like tupac he was a brother too...fuckin white boys get things right next time.,

PREDICTED CLASSES:
['clean']
--------------------------------------------------
TEXT:
== From RfC == 

 The title is fine as it is, IMO.

PREDICTED CLASSES:
['clean']
--------------------------------------------------
TEXT:
" 

 == Sources == 

 * Zawe Ashton on Lapland —  /  "

PREDICTED CLASSES:
['clean']
--------------------------------------------------
TEXT:
:If you have a look back at the source, the information I updated was the correct form. I can only guess the source hadn't updated. I shall update the information once again but thank you for your message.

PREDICTE

In [ ]:
import joblib

# Save tokenizer
joblib.dump(tokenizer, "tokenizer.pkl")

# Save stopwords and lemmatizer
joblib.dump(stop_words, "stopwords.pkl")
joblib.dump(lemmatizer, "lemmatizer.pkl")

# Save model
topic_model.save("toxic_comment_model.keras")

print("All files saved successfully!")

All files saved successfully!


In [ ]:
from google.colab import files

# Download files one by one
files.download("tokenizer.pkl")
files.download("stopwords.pkl")
files.download("lemmatizer.pkl")
files.download("toxic_comment_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>